In [15]:
import csv
import os

# Define the input file path
input_file_path = r"C:\Users\abhis\OneDrive\sem 4\BIO\Parjet\output1.csv"

# Define the target locations
target_locations = ['California', 'Texas', 'New York']

# Function to detect the delimiter in the CSV file
def detect_delimiter(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        first_line = file.readline().strip()
        
    for delimiter in [',', '\t', ';']:
        if delimiter in first_line:
            parts = first_line.split(delimiter)
            # If splitting produces multiple parts and they look like headers
            if len(parts) > 1 and any('ID' in part or 'sequence' in part.lower() or 'location' in part.lower() for part in parts):
                return delimiter
    
    # Default to comma if unable to detect
    return ','

try:
    # Detect the delimiter
    delimiter = detect_delimiter(input_file_path)
    print(f"Detected delimiter: '{delimiter}'")
    
    # Read the CSV file
    with open(input_file_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=delimiter)
        
        # Read header
        headers = next(reader)
        print(f"Headers found: {headers}")
        
        # Standardize column indices
        col_indices = {
            'Sequence ID': -1,
            'Nucleotide Sequence': -1,
            'location': -1,
            'host': -1,
            'releaseDate': -1
        }
        
        # Find column indices based on header names
        for i, header in enumerate(headers):
            header_lower = header.lower()
            if 'sequence id' in header_lower or 'accession' in header_lower:
                col_indices['Sequence ID'] = i
            elif 'nucleotide sequence' in header_lower or ('sequence' in header_lower and 'id' not in header_lower):
                col_indices['Nucleotide Sequence'] = i
            elif 'location' in header_lower:
                col_indices['location'] = i
            elif 'host' in header_lower:
                col_indices['host'] = i
            elif 'release' in header_lower or 'date' in header_lower:
                col_indices['releaseDate'] = i
        
        print(f"Column indices: {col_indices}")
        
        # Create dictionaries to store data for each location
        location_data = {loc: [] for loc in target_locations}
        
        # Process each row
        for row in reader:
            # Check if this row has a location field
            if col_indices['location'] >= 0 and col_indices['location'] < len(row):
                location_value = row[col_indices['location']]
                
                # Check each target location
                for location in target_locations:
                    if location.lower() in location_value.lower():
                        # Create a dictionary for this row with standardized columns
                        row_data = {}
                        for col_name, col_idx in col_indices.items():
                            if col_idx >= 0 and col_idx < len(row):
                                row_data[col_name] = row[col_idx]
                            else:
                                row_data[col_name] = ""  # Empty value for missing columns
                        
                        # Add to the corresponding location
                        location_data[location].append(row_data)
        
        # Create output directory in the same location as the input file
        output_dir = os.path.dirname(input_file_path)
        
        # Save each location data to a CSV file
        for location, data in location_data.items():
            if data:  # Only create file if there's data for this location
                filename = os.path.join(output_dir, f"{location.replace(' ', '_')}_data.csv")
                
                with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
                    writer = csv.DictWriter(csvfile, fieldnames=list(col_indices.keys()))
                    writer.writeheader()
                    writer.writerows(data)
                
                print(f"Created {filename} with {len(data)} entries")
            else:
                print(f"No data found for {location}")

except Exception as e:
    print(f"Error processing file: {e}")
    import traceback
    traceback.print_exc()

print("Data extraction complete!")

Detected delimiter: ','
Headers found: ['Feature', 'Value', 'Sequence ID', 'Nucleotide Sequence', 'biosample', 'isLabHost', 'virus', 'releaseDate', 'host', 'geneCount', 'submitter', 'labHost', 'isolate', 'proteinCount', 'updateDate', 'length', 'bioprojects', 'purposeOfSampling', 'nucleotide', 'isAnnotated', 'sourceDatabase', 'completeness', 'maturePeptideCount', 'sraAccessions', 'location']
Column indices: {'Sequence ID': 23, 'Nucleotide Sequence': 3, 'location': 24, 'host': 11, 'releaseDate': 14}
Created C:\Users\abhis\OneDrive\sem 4\BIO\Parjet\California_data.csv with 1092 entries
Created C:\Users\abhis\OneDrive\sem 4\BIO\Parjet\Texas_data.csv with 763 entries
Created C:\Users\abhis\OneDrive\sem 4\BIO\Parjet\New_York_data.csv with 212 entries
Data extraction complete!
